In [1]:
from src.data import Dataloader
import pandas as pd
import os
from src.utils import seed_everything
import random
from tqdm import tqdm
from typing import Callable
import torch.nn.functional as F
from src.data import TimeSeriesDataset
import torch
from src.nn import MLP, SlidingWindowBinaryClassification, SlidingWindowRegression
from sklearn.metrics import roc_auc_score, precision_score
import warnings
from sklearn.exceptions import DataConversionWarning

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from src.nn import RNN, GRU, LSTM

warnings.filterwarnings("ignore", category=DataConversionWarning)


## 1. Set up
To set up experience environment, we perform the following steps:

1. This section below will set up `const` needed for experiment
2. For reproducibility, we also set the same inital seed for everything (`numpy`, `torch`, etc.)

In [3]:
"""
 Step 1.  This section below will set up `const` needed for experiment
"""
# Clean data location
DATA_PATH = os.path.abspath('data/clean/')

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15  

# Columns that are used as features for training
FEAT_COLUMN = ['Close','High','Low', 'Open','Volume','sentiment_score_mean','sentiment_polarity_mean','sentiment_subjectivity_mean','news_count', 'log_return', 'vol_7d','vol_30d','ma_7','ma_30','ma_ratio']

# Columns that are used as features for training but do NOT need to normalize
NOT_NORMALIZE_FEAT = ['sentiment_score_mean','sentiment_polarity_mean','sentiment_subjectivity_mean','news_count','ma_ratio','log_return']

# Label for binary classification task
BINARY_LABEL_COLUMN = 'price_increase'

# Label for regression task
REGRESSION_LABEL_COLUMN = 'next_close'

# Number of tokens used in training to demonstrate scaling laws
LOG_SCALING = [2,4,8,16,32]

# For reproducibility, inital random seed
INIT_SEED = 720

"""
 Step 2. For reproducibility, we also set the same inital seed for everything (`numpy`, `torch`, etc.)
"""
seed_everything(INIT_SEED)

## 2. Prepare dataset for training

1. Shuffle all token networks (I.I.D)
2. Split datasets for training, validation and testing: the first 70% of datasets for training, the next 15% for validation, the next 15% for testing
3. Load all dataset in memory to `TimeSeriesDataset`
4. For each token, we use all data before `2023-12-20` for trainning, from `2023-12-20` to `2024-12-20` for validation and `2024-12-20` onward for testing

There are 49 tokens datasets in total, so we keep 32 networks for training, 17 tokens for validation and 16 tokens for testing.

We first load a single token dataset as an example

In [5]:
data_loader = Dataloader(DATA_PATH)
timeseries_data = data_loader.from_csv("aave.csv", feat_columns = FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN)

train_data_example, val_data_example, test_data_example = timeseries_data.split_by_ratio(TRAIN_RATIO,VAL_RATIO,TEST_RATIO)

print(train_data_example.x.shape)
print(val_data_example.x.shape)
print(test_data_example.x.shape)

# Retrieve feature map for late usage
FEAT_MAP = train_data_example.feat_map
print(FEAT_MAP)

torch.Size([541, 15])
torch.Size([116, 15])
torch.Size([116, 15])
{'Close': 0, 'High': 1, 'Low': 2, 'Open': 3, 'Volume': 4, 'sentiment_score_mean': 5, 'sentiment_polarity_mean': 6, 'sentiment_subjectivity_mean': 7, 'news_count': 8, 'log_return': 9, 'vol_7d': 10, 'vol_30d': 11, 'ma_7': 12, 'ma_30': 13, 'ma_ratio': 14}


In [7]:
seed_everything(INIT_SEED)
files = os.listdir(DATA_PATH)
random.shuffle(files) # Step 1

# Step 2
train_token_list = files[:32]
valid_token_list = files[32:33+8]
test_token_list = files [-8:]

display(train_token_list)


assert len(set(train_token_list).intersection(set(valid_token_list))) == 0
assert len(set(test_token_list).intersection(set(valid_token_list))) == 0

# Step 3
data_loader = Dataloader(DATA_PATH)

train_data = []
valid_data = []
test_data = []

for file_name in tqdm(train_token_list):
    train_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN))

for file_name in tqdm(valid_token_list):
    valid_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN))

for file_name in tqdm(test_token_list):
    test_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN))

FEAT_MAP = train_data[0].feat_map
train_data_split = []
valid_data_split = []
test_data_split = []

for data in tqdm(train_data):
    train, val,test = data.split_by_ratio(TRAIN_RATIO,VAL_RATIO,TEST_RATIO)
    train_data_split.append((train,val,test))

for data in tqdm(valid_data):
    train, val,test = data.split_by_ratio(TRAIN_RATIO,VAL_RATIO,TEST_RATIO)
    valid_data_split.append((train,val,test))

for data in tqdm(test_data):
    train, val,test = data.split_by_ratio(TRAIN_RATIO,VAL_RATIO,TEST_RATIO)
    test_data_split.append((train,val,test))




['algorand.csv',
 'immutable.csv',
 'litecoin.csv',
 'chainlink.csv',
 'toncoin.csv',
 'dogecoin.csv',
 'polygon.csv',
 'arbitrum.csv',
 'avalanche.csv',
 'binance_coin.csv',
 'flow.csv',
 'tezos.csv',
 'stellar.csv',
 'sandbox.csv',
 'uniswap.csv',
 'theta.csv',
 'filecoin.csv',
 'injective.csv',
 'render.csv',
 'the_graph.csv',
 'usd_coin.csv',
 'lido.csv',
 'bitcoin_cash.csv',
 'fantom.csv',
 'eos.csv',
 'kaspa.csv',
 'ethereum.csv',
 'xrp.csv',
 'maker.csv',
 'cosmos.csv',
 'bitcoin.csv',
 'tether.csv']

100%|██████████| 8/8 [00:00<?, ?it/s]


In [8]:
train_shape = {}
valid_shape = {}
test_shape = {}

for example_data_split in [train_data_split,valid_data_split,test_data_split]:
    for train ,val, test in example_data_split:
        train_shape[tuple(train.x.shape)] = train_shape.get(tuple(train.x.shape),0) + 1
        valid_shape[tuple(val.x.shape)] = valid_shape.get(tuple(val.x.shape),0) + 1
        test_shape[tuple(test.x.shape)] = test_shape.get(tuple(test.x.shape),0) + 1


# train_data_example, val_data_example,test_data_example =  train_data_split[3]
print(train_shape)
print(valid_shape)
print(test_shape)

{(541, 15): 41, (533, 15): 1, (190, 15): 1, (102, 15): 1, (396, 15): 1, (161, 15): 1, (298, 15): 1, (163, 15): 1, (428, 15): 1}
{(116, 15): 41, (114, 15): 1, (41, 15): 1, (22, 15): 1, (85, 15): 1, (34, 15): 1, (64, 15): 1, (35, 15): 1, (92, 15): 1}
{(116, 15): 41, (115, 15): 1, (40, 15): 1, (22, 15): 1, (85, 15): 1, (35, 15): 2, (64, 15): 1, (92, 15): 1}


4. Next we normalize each feature with min and max of each feature

In [9]:
def normalize(train_data: TimeSeriesDataset, valid_data: TimeSeriesDataset, test_data: TimeSeriesDataset, normalize_y = False, exclude_cols = []):
    n_cols = train_data.x.shape[1]
    include_cols = [i for i in range(n_cols) if i not in (exclude_cols or [])]

    max_values, _ = torch.max(train_data.x[:, include_cols], dim=0)
    min_values, _ = torch.min(train_data.x[:, include_cols], dim=0)

    denom = (max_values - min_values)
    denom[denom == 0] = 1  # avoid division by zero

    for data in [train_data, valid_data, test_data]:
        data.x[:, include_cols] = (data.x[:, include_cols] - min_values) / denom

    if normalize_y:
        max_y, _ = torch.max(train_data.y, dim=0)
        min_y, _ = torch.min(train_data.y, dim=0)
        for data in [train_data, valid_data, test_data]:
            data.y = (data.y - min_y) / (max_y - min_y)

    return train_data, valid_data, test_data


normalize_feat_exclude_idx = [idx for feat,idx in FEAT_MAP.items() if feat in NOT_NORMALIZE_FEAT]
for data_split in [train_data_split, valid_data_split, test_data_split]:
    for train,val, test in data_split:
        normalize(train,val,test,exclude_cols = normalize_feat_exclude_idx)
        


In [10]:
first_train, first_val, fist_test = train_data_split[0]
first_train.x[0]

tensor([0.7957, 0.5960, 0.7722, 0.7836, 0.0937, 1.0000, 0.1600, 0.5000, 1.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])

## 3. Traditional machine learning baseline

In this project, we explore the following traditional machine learning for binary classification
1. Logistic Regression
2. Random Forest
3. Gradient Boosting (GBM)
4. SVM
5. KNN

In [ ]:
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

def traditional_model_experiment(n_tokens, train_data,val_data, test_data):
    used_train_data = train_data[:n_tokens]
    used_val_data = val_data[:n_tokens]
    all_train_x = []
    all_train_y = []
    all_test_x = []
    all_test_y = []

    for train, val, test in used_train_data:
        all_train_x.append(train.x)
        all_train_y.append(train.y)
        
        # Since we don't need validation set for traditional ML models, we use them for training
        all_train_x.append(val.x)
        all_train_y.append(val.y)
    
    for train, val, test in used_val_data:
        all_train_x.append(train.x)
        all_train_y.append(train.y)
        
        # Since we don't need validation set for traditional ML models, we use them for training
        all_train_x.append(val.x)
        all_train_y.append(val.y)


        
    for train, _, test in test_data: 
        all_test_y.append(test.y)
        all_test_x.append(test.x)

    train_x = torch.concatenate(all_train_x).numpy()
    train_y = torch.concatenate(all_train_y).numpy()
    test_x = torch.concatenate(all_test_x).numpy()
    test_y = torch.concatenate(all_test_y).squeeze().numpy()

    models = {
        "Logistic Regression":     LogisticRegression(),
        "Random Forest":           RandomForestClassifier(),
        "Gradient Boosting (GBM)": GradientBoostingClassifier(),
        "SVM":                     SVC(probability=True),
        "KNN":                     KNeighborsClassifier(),
    }
    results = {}
    results['# Training tokens'] = n_tokens
    for name, model in tqdm(models.items()):
        model.fit(train_x, train_y)
        preds = model.predict_proba(test_x)[:, 1]
        auc = roc_auc_score(test_y, preds)
        results[name] = auc

    return results

all_result = []
for n in LOG_SCALING:
    n_result = traditional_model_experiment(n,train_data_split,valid_data_split,test_data_split[:1] )
    all_result.append(n_result)

traditional_results_df = pd.DataFrame(all_result)
traditional_results_df


  0%|          | 0/5 [00:00<?, ?it/s]d:\anaconda\envs\data2010\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
  0%|          | 0/5 [00:00<?, ?it/s]d:\anaconda\envs\data2010\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
 

4. Sliding window baseline

In [11]:

auc = []
for data in test_data_split:
    baseline = SlidingWindowBinaryClassification()
    train, val, test = data
    preds = []

    baseline.update(val.y[-1].item())
    for _,_,y in test:
        y = baseline()
        preds.append(y)
        baseline.update(y) 

    score = roc_auc_score(test.y.squeeze().tolist(), preds)
    auc.append(score)

sliding_window_result_df = pd.DataFrame([{'Sliding Window - Binary Classification': sum(auc)/len(auc)}])
sliding_window_result_df

,Sliding Window - Binary Classification
0,0.5


## Training loop

In [18]:

import copy
data_loader = Dataloader(DATA_PATH)
timeseries_data = data_loader.from_csv("aptos.csv", feat_columns = FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN)

train_data, val_data,test_data = timeseries_data.split_by_ratio(TRAIN_RATIO, VAL_RATIO, TEST_RATIO)

normalize_feat_exclude_idx = [idx for feat,idx in FEAT_MAP.items() if feat in NOT_NORMALIZE_FEAT]

train_data, val_data, test_data = normalize(train_data,val_data, test_data, exclude_cols=normalize_feat_exclude_idx)

def evaluate(data: TimeSeriesDataset, model: torch.nn.Module, evaluator: Callable = roc_auc_score):
    model.eval()
    preds = []
    with torch.no_grad():
        for time, feat, y in data:
            pred = model(feat).sigmoid()
            
            preds.append(pred.item())
    return evaluator(data.y.squeeze().tolist(), preds)

def train(train_data : TimeSeriesDataset, val_data : TimeSeriesDataset, model: torch.nn.Module, criterion : Callable, opt: torch.optim.Optimizer, epoch = 100):
    
    best_val = -1
    best_model_state_dict = copy.deepcopy(model.state_dict())
    for i in range(epoch):
        model.train()
        all_loss = []
        preds = []
        for time, feat, y in train_data:
            opt.zero_grad()
            z = model(feat.float())
            loss = criterion(
                z.float(),y.float()
            )
            
            all_loss.append(loss.item())
            loss.backward()
            opt.step()

            preds.append(z.sigmoid().item())
        
        auc = roc_auc_score(train_data.y.squeeze().tolist(), preds)
        valid_auc = evaluate(val_data,model)
        if valid_auc > best_val:
            best_val = valid_auc
            best_model_state_dict = copy.deepcopy(model.state_dict())
        print(f"[INFO] Epoch - {i+1} : Loss - {sum(all_loss)/float(len(all_loss))} ; Train-AUC : {auc}; Validation-AUC: {valid_auc}")
    
    model.load_state_dict(best_model_state_dict)
    return model

def multiple_token_training(training_tokens: list[TimeSeriesDataset], model:torch.nn.Module,criterion : Callable, opt: torch.optim.Optimizer, epoch = 50):
    for train_data, val_data, test_data in training_tokens:
        train(train_data,val_data,model,criterion,opt,epoch)
        evaluate(test_data,model)


model = MLP(in_channel= len(FEAT_COLUMN), out_channel= 1, dim = 64, num_layers=3,dropout=0.5)

opt = torch.optim.Adam(
    model.parameters(), lr=float(0.0003)
)

score = evaluate(test_data,model)
print(score)

model = train(train_data, val_data, model, F.binary_cross_entropy_with_logits,opt)

score = evaluate(test_data,model)
print(score)


           


0.5376344086021505
[INFO] Epoch - 1 : Loss - 0.8330645407706299 ; Train-AUC : 0.437998647734956; Validation-AUC: 0.5757575757575758
[INFO] Epoch - 2 : Loss - 0.7298912812999431 ; Train-AUC : 0.5134099616858238; Validation-AUC: 0.6705767350928641
[INFO] Epoch - 3 : Loss - 0.7254913477289596 ; Train-AUC : 0.5016452558034707; Validation-AUC: 0.5845552297165201
[INFO] Epoch - 4 : Loss - 0.7075075661176004 ; Train-AUC : 0.5050709939148073; Validation-AUC: 0.4594330400782014
[INFO] Epoch - 5 : Loss - 0.7065578754116225 ; Train-AUC : 0.4827135451881903; Validation-AUC: 0.4301075268817205
[INFO] Epoch - 6 : Loss - 0.6931172625330471 ; Train-AUC : 0.5437457741717376; Validation-AUC: 0.43206256109481916
[INFO] Epoch - 7 : Loss - 0.701290110013629 ; Train-AUC : 0.506288032454361; Validation-AUC: 0.5650048875855328
[INFO] Epoch - 8 : Loss - 0.6916495050559908 ; Train-AUC : 0.5437908496732027; Validation-AUC: 0.5747800586510264
[INFO] Epoch - 9 : Loss - 0.7051889547365624 ; Train-AUC : 0.4344827586

In [ ]:
from src.nn import RNN, GRU, LSTM
rnn = RNN(len(FEAT_COLUMN),1,128,4)


for time,x,y in train_data:
    x = x.unsqueeze(0).unsqueeze(0)
    out, h = rnn(x, None)
    print(out.shape)
    break

ValueError: RNN: Expected input to be 2D or 3D, got 1D tensor instead

# Regression

In [5]:
train_data = []
valid_data = []
test_data = []

for file_name in tqdm(train_token_list):
    train_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=REGRESSION_LABEL_COLUMN))

for file_name in tqdm(valid_token_list):
    valid_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=REGRESSION_LABEL_COLUMN))

for file_name in tqdm(test_token_list):
    test_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=REGRESSION_LABEL_COLUMN))

train_data_split = []
valid_data_split = []
test_data_split = []

for data in tqdm(train_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    train_data_split.append((train,val,test))

for data in tqdm(valid_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    valid_data_split.append((train,val,test))

for data in tqdm(test_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    test_data_split.append((train,val,test))

for data_split in [train_data_split, valid_data_split, test_data_split]:
    for train,val, test in data_split:
        normalize(train,val,test,normalize_y=True)


100%|██████████| 8/8 [00:00<00:00, 1450.38it/s]


In this project, we explore the following traditional machine learning for regression
1. Linear Regression
2. Ridge
3. Lasso
4. Random Forest
5. Gradient Boosting (GBM)
6. SVR

In [ ]:

def traditional_regression_experience(n_tokens, train_data, test_data):
    used_train_data = train_data[:n_tokens]
    all_train_x = []
    all_train_y = []
    all_test_x = []
    all_test_y = []

    for train, _, test in used_train_data:
        all_train_x.append(train.x)
        all_train_y.append(train.y)  # ← change to your regression label attribute
        
    for train, _, test in test_data: 
        all_test_y.append(test.y)
        all_test_x.append(test.x)

    train_x = torch.concatenate(all_train_x).numpy()
    train_y = torch.concatenate(all_train_y).squeeze().numpy()
    test_x  = torch.concatenate(all_test_x).numpy()
    test_y  = torch.concatenate(all_test_y).squeeze().numpy()

    models = {
        "Linear Regression":       LinearRegression(),
        "Ridge":                   Ridge(alpha=1.0),
        "Lasso":                   Lasso(alpha=0.1),
        "Random Forest":           RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting (GBM)": GradientBoostingRegressor(n_estimators=100, random_state=42),
        "SVR":                     SVR(),
    }

    results = {}
    results['# Training tokens'] = n_tokens
    for name, model in tqdm(models.items()):
        model.fit(train_x, train_y)
        preds = model.predict(test_x)
        results[f"{name} RMSE"] = np.sqrt(mean_squared_error(test_y, preds))
        results[f"{name} R²"]   = r2_score(test_y, preds)

    return results

all_result = []
for n in LOG_SCALING:
    n_result = traditional_regression_experience(n, train_data_split, test_data_split[:1])
    all_result.append(n_result)

traditional_regression_df = pd.DataFrame(all_result)
traditional_regression_df

100%|██████████| 6/6 [00:23<00:00,  3.84s/it]


,# Training tokens,Linear Regression RMSE,Linear Regression R²,Ridge RMSE,Ridge R²,Lasso RMSE,Lasso R²,Random Forest RMSE,Random Forest R²,Gradient Boosting (GBM) RMSE,Gradient Boosting (GBM) R²,SVR RMSE,SVR R²
0,2,0.004103,0.974075,0.004254,0.972128,0.151525,-34.363545,0.013309,0.727174,0.014418,0.679821,0.099156,-14.143514
1,4,0.004010,0.975235,0.004049,0.974746,0.172541,-44.853276,0.013603,0.715003,0.014897,0.658203,0.098850,-14.050006
2,8,0.004619,0.967144,0.004011,0.975223,0.215627,-70.613535,0.014910,0.657600,0.017499,0.528362,0.090894,-11.724984
3,16,0.004143,0.973568,0.003947,0.976006,0.200242,-60.758354,0.013651,0.712978,0.015119,0.647932,0.097154,-13.538001
4,32,0.003924,0.976284,0.003942,0.976066,0.238749,-86.795316,0.013955,0.700045,0.015221,0.643141,0.092262,-12.110816


In [ ]:
rmse_scores = []
r2_scores = []

for data in test_data_split:
    baseline = SlidingWindowRegression()
    train, val, test = data
    preds = []

    baseline.update(val.y[-1].item())  # warm up with last val value
    for _, _, y in test:
        pred = baseline()
        preds.append(float(pred))
        baseline.update(y)

    rmse = np.sqrt(mean_squared_error(test.y.squeeze().tolist(), preds))
    r2   = r2_score(test.y.squeeze().tolist(), preds)
    rmse_scores.append(rmse)
    r2_scores.append(r2)

sliding_window_regression_df = pd.DataFrame([{
    'Sliding Window - Regression RMSE': sum(rmse_scores) / len(rmse_scores),
    'Sliding Window - Regression R²':   sum(r2_scores)   / len(r2_scores),
}])
sliding_window_regression_df

,Sliding Window - Regression RMSE,Sliding Window - Regression R²
0,0.050803,0.934662


In [11]:
# 1. Check label balance
labels = train_data.y.squeeze().tolist()
print(f"Positive rate: {sum(labels)/len(labels):.4f}")  # should be near 0.5

# 2. Check model output variance BEFORE training
model.eval()
outs = []
with torch.no_grad():
   for time, feat, y  in train_data:
        z = model(feat.float()).sigmoid().item()
        outs.append(z)

import numpy as np
print(f"Output mean: {np.mean(outs):.4f}")
print(f"Output std:  {np.std(outs):.4f}")  # near 0 = model outputs same thing for all inputs

# 3. Check feature variance after normalization
print(f"Feature mean: {train_data.x.mean(dim=0)}")
print(f"Feature std:  {train_data.x.std(dim=0)}")  # any column near 0 = dead feature

# 4. Check sizes
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")
print(f"Train labels: {train_data.y.shape}")
print(f"Train features: {train_data.x.shape}")


Positive rate: 0.5006
Output mean: 0.5111
Output std:  0.0685
Feature mean: tensor([0.2086, 0.1568, 0.2130, 0.2095])
Feature std:  tensor([0.2333, 0.1765, 0.2308, 0.2344])
Train: 799, Val: 366, Test: 381
Train labels: torch.Size([799, 1])
Train features: torch.Size([799, 4])


In [ ]:
from typing import Dict
def evaluate(data: TimeSeriesDataset, model: torch.nn.Module, evaluator: Callable = roc_auc_score):
    model.eval()
    preds = []
    with torch.no_grad():
        for time, feat, y in data:
            pred = model(feat.float()).sigmoid()
            
            preds.append(pred.item())
    return evaluator(data.y.squeeze().tolist(), preds)

def train_pipeline(
        train_data : TimeSeriesDataset, 
        val_data : TimeSeriesDataset, 
        model: torch.nn.Module, 
        criterion : Callable, 
        opt: torch.optim.Optimizer, 
        epoch = 100, 
        patience =  20,
        evaluator: Callable = roc_auc_score
    ):
    
    best_val = -1
    best_model_state_dict = copy.deepcopy(model.state_dict())
    patience_count = 0
    for i in range(epoch):
        model.train()
        all_loss = []
        preds = []
        for time, feat, y in train_data:
            opt.zero_grad()
            z = model(feat.float())
            loss = criterion(
                z.float(),y.float()
            )
            
            all_loss.append(loss.item())
            loss.backward()
            opt.step()

            preds.append(z.sigmoid().item())
        
        auc = evaluator(train_data.y.squeeze().tolist(), preds)
        valid_auc = evaluate(val_data,model,evaluator=evaluator)
        if valid_auc > best_val:
            patience_count = 0
            best_val = valid_auc
            best_model_state_dict = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stoppting at epoch {i}")
                break
        print(f"[INFO] Epoch - {i+1} : Loss - {sum(all_loss)/float(len(all_loss))} ; Train-AUC(for binary classification)/MSE(for regression) : {auc}; Validation-AUC(for binary classification)/MSE(for regression): {valid_auc}")
    
    model.load_state_dict(best_model_state_dict)
    return model

def nn_experiment(train_data: TimeSeriesDataset,val_data: TimeSeriesDataset,test_data: TimeSeriesDataset, regression = False):
    if regression:
        criterion = F.mse_loss
        metric = mean_absolute_percentage_error
    else:
        criterion = F.binary_cross_entropy_with_logits
        metric = roc_auc_score
    model = MLP(in_channel= len(ALL_FEAT_COLUMN), out_channel= 1, dim = 64, num_layers=3,dropout=0.5)

    opt = torch.optim.Adam(
        model.parameters(), lr=float(0.0003)
    )
    model = train_pipeline(train_data, val_data, model, criterion, opt, evaluator=metric)
    test_result = evaluate(test_data,model,evaluator = metric)
    return test_result


task = REGRESSION_LABEL_COLUMN
seed_everything(INIT_SEED)

result_file_name = "auc_mlp_binary.csv"
result_path_file = os.path.join(RESULT_PATH,result_file_name)
if os.path.isfile(result_path_file) and not RE_RUN_EVERYTHING:
    arima_df = pd.read_csv(result_path_file)
    print("INFO: LOADED existing results")

else:
    print("INFO: Re-computing results")
    all_data_split = prepare_datasets(files, ALL_FEAT_COLUMN, task)

    all_rows_arima = []
    for train, val, test in tqdm(all_data_split):
        mape = nn_experiment(train,val,test, regression=True)
        all_rows_arima.append({'Dataset' : train.dataset_name, 'MAPE': mape})
    
    arima_df = pd.DataFrame(all_rows_arima)
    arima_df.to_csv(result_path_file, index= False)


arima_df.head(32)